# Monitor the job and check the result

After `case.submit`, your simulation is in the hands of the batch scheduler. This page shows how to follow the job while it waits and runs, and how to confirm that it finished successfully once it disappears from the queue.

## Monitor the job

Use `qstat` to see whether your job is queued or running:

```bash
qstat -u $USER
```

You may see output similar to this:

```text
                                                            Req'd  Req'd   Elap
Job ID          Username Queue    Jobname    SessID NDS TSK Memory Time  S Time
--------------- -------- -------- ---------- ------ --- --- ------ ----- - -----
6315297.desche* hannay   cpu      run.case01    --    6 768  1410g 12:00 Q   --
6315298.desche* hannay   cpu      st_archiv*    --    1   1  235gb 00:20 H   --
```

Submit does not mean "run immediately". The job first waits in the queue (`Q`). Once Derecho
starts it, the status changes to `R` and the elapsed time starts counting:

```text
6315297.desche* hannay   cpu      run.case01    --    6 768  1410g 12:00 R 00:00
6315298.desche* hannay   cpu      st_archiv*    --    1   1  235gb 00:20 H   --
```

When the job disappears from the list, it has either completed or crashed. The next section
shows how to tell which.

![The three qstat states: queued, running, gone](../../images/04.qstat_states.png)

CESM submits two jobs: `run.CASENAME` runs the simulation, and `st_archive` organizes the output after the simulation finishes.

The `S` column shows the job status. The most common status codes are:

| Status | Meaning |
|---|---|
| `Q` | The job is queued and waiting to run. |
| `R` | The job is running. |
| `H` | The job is being held while it waits for another job to finish. |
| Not listed | The job has finished or stopped because of an error. |

A job disappearing from `qstat` does not necessarily mean that it succeeded.

## Check whether the simulation succeeded

### 1. Check `CaseStatus`

CESM records the status of each step in a file named `CaseStatus`, located in the case directory.

Use:

```text
tail -20 ~/cases/CASENAME/CaseStatus
```

A successful simulation includes:

```text
case.run success
```

After CESM organizes the output files, you should also see:

```text
st_archive success
```

If you see `case.run error`, the simulation crashed. The log files will provide more information about what went wrong.

### 2. Check the CESM log

You can also search the main CESM log for a successful-termination message:

```text
zgrep "SUCCESSFUL TERMINATION" ~/cases/CASENAME/logs/cesm.log.*.gz
```

A successful run should contain a line similar to:

```text
(seq_mct_drv): =============== SUCCESSFUL TERMINATION OF CPL7-cesm ===============
```

If you do not find this message, examine the log files for an error or traceback. The [Troubleshooting](../09.Troubleshooting/9.1_log_files_and_crashes.ipynb) chapter explains what to check when a run crashes.

### 3. Check the archive

After `st_archive success` appears, list the archive directory:

```text
ls /glade/derecho/scratch/$USER/archive/CASENAME
```

You should see directories containing model output, restart files, and log files.

## How the five directories are used

| Directory | Role in the workflow |
|---|---|
| **CESM code** | Provides the source code and the `create_newcase` command. |
| **Input data** | Provides the grids, initial conditions, and forcing data required by the selected case. |
| **Case** | Is created by `create_newcase` and contains the experiment configuration and commands. |
| **Build/run** | Is used when CESM is compiled and when the simulation runs. |
| **Archive** | Receives organized output, restart, and log files after the run. |

![The CESM directories after a run completes](../../images/04.cesm_after_run_complete.png)

In the next exercise, you will apply this workflow to create and run a specific CESM case.